# Desafio 1: Prevenindo Data Leakage com Pipelines

O problema aqui é o Data Leakage (Vazamento de Dados). Se usarmos o `fit_transform` antes de separar os dados (`train_test_split`), o TF-IDF vai usar os textos de teste para calcular os pesos das palavras. Assim, o modelo "trapaceia" vendo dados do futuro.

A solução padrão é usar um `Pipeline`. Ele garante que o `fit` rode apenas no `X_train`, validando o modelo de forma correta e sem vazamentos.

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

reviews = ["Produto ótimo", "Péssimo", "Excelente, adorei", "Quebrado e ruim", "Muito bom"]
labels = [1, 0, 1, 0, 1]

# 1. O divisor de águas: Divisão ANTES de qualquer processamento de texto
X_train, X_test, y_train, y_test = train_test_split(reviews, labels, test_size=0.4, random_state=42)

# 2. Criação do Pipeline (Industry Standard) para prevenir o Vazamento
pipeline_desafio1 = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

# 3. O pipeline encapsula o .fit() do vetorizador e restringe exclusivamente aos dados X_train
pipeline_desafio1.fit(X_train, y_train)

# 4. Avaliação (O X_test passará apenas por um 'transform' isolado dentro do pipeline)
score = pipeline_desafio1.score(X_test, y_test)
print(f"Acurácia real sem vazamento: {score:.2f}")

Acurácia real sem vazamento: 0.50


# Desafio 2: Capturando Contexto com N-gramas

Modelos clássicos perdem o contexto de inversões como "não gostei" porque olham as palavras isoladas. 

Quando ativamos o `ngram_range=(1,2)`, o modelo passa a ler "não gostei" como um único token (bigrama). Isso permite ao Naive Bayes entender que essa dupla tem forte peso negativo.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

textos = ["O produto é bom", "Não é bom", "Gostei muito", "Não gostei do material"]
y = [1, 0, 1, 0]

# 1. Instanciando Pipeline com suporte a N-gramas (1 a 2 palavras)
pipeline_desafio2 = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('clf', MultinomialNB())
])
pipeline_desafio2.fit(textos, y)

# 2. Predição para a frase de teste traiçoeira
frase_teste = ["Não gostei, mas dizem que é bom"]
predicao = pipeline_desafio2.predict(frase_teste)

print(f"Frase a testar: '{frase_teste[0]}'")
print(f"Predição Capturada: {'Positivo' if predicao[0] == 1 else 'Negativo'}")

# 3. Interpretando e Extraindo a Importância dos n-gramas capturados (Log Probs)
tfidf = pipeline_desafio2.named_steps['tfidf']
clf = pipeline_desafio2.named_steps['clf']
features = tfidf.get_feature_names_out()

print("\n--- Features Preditivas Mais Fortes para a Classe Negativa (0) ---")
top_indices_negativos = clf.feature_log_prob_[0].argsort()[-5:][::-1]
for idx in top_indices_negativos:
    print(f"Feature Semântica: '{features[idx]}' -> (Log Probabilidade: {clf.feature_log_prob_[0][idx]:.4f})")

Frase a testar: 'Não gostei, mas dizem que é bom'
Predição Capturada: Negativo

--- Features Preditivas Mais Fortes para a Classe Negativa (0) ---
Feature Semântica: 'não' -> (Log Probabilidade: -2.2429)
Feature Semântica: 'não bom' -> (Log Probabilidade: -2.3423)
Feature Semântica: 'bom' -> (Log Probabilidade: -2.4308)
Feature Semântica: 'material' -> (Log Probabilidade: -2.5171)
Feature Semântica: 'não gostei' -> (Log Probabilidade: -2.5171)


# Desafio 3: Lado Prático de Dados Desbalanceados

Trabalhar com dados muito desbalanceados (ex: 95% positivo e 5% negativo) num split aleatório pode gerar um conjunto de teste sem nenhum exemplo da classe minoritária. Isso causa erro de divisão por zero (`UndefinedMetricWarning`) no cálculo de métricas como precisão e recall.

Usar o `stratify=y` resolve isso, forçando o Scikit-Learn a manter a mesma proporção exata de positivos e negativos no treino e no teste.

*Nota técnica:* Se a classe minoritária tiver só 1 exemplo, a matemática do split quebra. Por isso os dados abaixo foram duplicados.

In [3]:
from sklearn.model_selection import train_test_split
from collections import Counter

X_desbalanceado = ["Amei", "Perfeito", "Excelente", "Muito bom", "Ótimo produto", 
                   "Recomendo", "Chegou rápido", "Nota 10", "Qualidade boa", "Péssimo e estragado"]
y_desbalanceado = [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]

# Como o scikit-learn exige pelo menos 2 ocorrências da classe minoritária para estratificar,
# estamos duplicando a amostra para viabilizar e explicitar o impacto da demonstração.
X_ampliado = X_desbalanceado * 2
y_ampliado = y_desbalanceado * 2

# Divisão Aleatória Padrão (Vulnerável)
X_train_ruim, X_test_ruim, y_train_ruim, y_test_ruim = train_test_split(
    X_ampliado, y_ampliado, test_size=0.25, random_state=42
)

# Divisão Estratificada (Arquitetura Resiliente)
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X_ampliado, y_ampliado, test_size=0.25, random_state=42, stratify=y_ampliado
)

print("================ ANÁLISE DE ESTRATIFICAÇÃO ================")
print(f"Distribuição da Base Completa  : {dict(Counter(y_ampliado))}")

print("\n[X] COMPORTAMENTO SEM STRATIFY")
print(f" - Distribuição em Treino : {dict(Counter(y_train_ruim))}")
print(f" - Distribuição em Teste  : {dict(Counter(y_test_ruim))}  <- Notem o enviesamento aleatório")

print("\n[V] COMPORTAMENTO COM STRATIFY")
print(f" - Distribuição em Treino : {dict(Counter(y_train_strat))}")
print(f" - Distribuição em Teste  : {dict(Counter(y_test_strat))}  <- Respeitando precisamente a proporção global")

================ ANÁLISE DE ESTRATIFICAÇÃO ================
Distribuição da Base Completa  : {1: 18, 0: 2}

[X] COMPORTAMENTO SEM STRATIFY
 - Distribuição em Treino : {1: 13, 0: 2}
 - Distribuição em Teste  : {1: 5}  <- Notem o enviesamento aleatório

[V] COMPORTAMENTO COM STRATIFY
 - Distribuição em Treino : {1: 14, 0: 1}
 - Distribuição em Teste  : {0: 1, 1: 4}  <- Respeitando precisamente a proporção global
